In [27]:
import os

In [28]:
%pwd

'D:\\PythonProjects\\E2E-ML-Project-with-mlflow'

In [29]:
os.chdir("D:\\PythonProjects\\E2E-ML-Project-with-mlflow")

In [30]:
%pwd

'D:\\PythonProjects\\E2E-ML-Project-with-mlflow'

In [31]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True) 
# to define vairables, we use dataclass decorator. frozen=True means that 
# the values of the variables cannot be changed after initialization.

class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [32]:
from mlProject import *
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

class configurationManager:
    def __init__(self, 
                 config_filepath=CONFIG_FILE_PATH, 
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )
        return data_ingestion_config

In [33]:
import os
import urllib.request as request
import zipfile
from mlProject import logger
from mlProject.utils.common import get_size

In [34]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
    
    def download_zip_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        if not os.path.exists(unzip_path):
            os.makedirs(unzip_path)
            with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
                zip_ref.extractall(unzip_path)
            logger.info(f"File extracted at: {unzip_path}")
        else:
            logger.info(f"Data already exists at: {unzip_path}")


In [35]:
try:
    config = configurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_zip_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-06-28 11:25:31,947: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-28 11:25:31,950: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-28 11:25:31,950: INFO: common: yaml file: config\schema.yaml loaded successfully]
[2026-06-28 11:25:31,950: INFO: common: created directory at: artifacts]
[2026-06-28 11:25:31,954: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-28 11:25:31,954: INFO: 1284007578: File already exists of size: ~ 25 KB]
[2026-06-28 11:25:31,962: INFO: 1284007578: File extracted at: artifacts\data_ingestion\unzipped]
